# 80 · Titanic — Setup e ingesta de datos

Este es el primer notebook de una demostración de cinco partes del ciclo de vida
completo de un modelo de *machine learning* en Databricks: ingesta, exploración, experimentación con
MLflow, registro del modelo en Unity Catalog, inferencia por lotes y en este caso, exportación de resultados para
hacer *submit* en la competencia de Kaggle *Titanic — Machine Learning from Disaster*.

### Objetivos de aprendizaje
- Crear el catálogo, el esquema y el *Volume* de Unity Catalog donde vivirá la demostración.
- Ingerir los archivos crudos de Kaggle (`train.csv`, `test.csv`) con un esquema explícito.
- Materializar la capa **bronze** del patrón Medallion como tablas Delta idempotentes.
- Validar el volumen de datos y el conteo de valores nulos antes de modelar.

## 1. Obtener el dataset y subirlo al Volume

### 1.1 Descarga desde Kaggle
1. Entra a `https://www.kaggle.com/competitions/titanic/data` (necesitas una cuenta gratuita de Kaggle).
2. Descarga **`train.csv`** (891 filas, incluye la etiqueta `Survived`) y **`test.csv`** (418 filas, sin etiqueta).
3. Opcionalmente descarga `gender_submission.csv` para ver el formato esperado de la entrega.

#### Archivo adjunto (en el campus virtual)

### 1.2 Sube los archivos al Volume (interfaz de Databricks)
El *Volume* se crea automáticamente en la celda siguiente, pero también puedes crearlo desde la interfaz:
(reemplazar por el catalogo correcto)
**Catalog → `big_data_ii_2025` → `spark_examples` → pestaña Volumes → Create volume → `titanic`**.

Subir el CSV y asegurarse que quede de la siguiente forma:
   - `/Volumes/big_data_ii_2025/spark_examples/titanic/train.csv`
   - `/Volumes/big_data_ii_2025/spark_examples/titanic/test.csv`

> **Nota:** en el panel lateral derecho, ícono ⚙ *Environment*, confirma que la
> *environment version* sea **4 o superior**. Las versiones previas no soportan `pyspark.ml` sobre serverless.

## 2. Parámetros

Todas las partes de esta demostración usan el mismo catálogo, esquema y *Volume*. Las tablas llevan el
prefijo `titanic_` porque `spark_examples` es un esquema compartido con otros ejemplos del curso.

In [0]:
CATALOGO = "big_data_ii_2025"
ESQUEMA = "spark_examples"
VOLUMEN = "titanic"
VOLUMEN_EXPERIMENTOS = "titanic_mlflow"

RUTA_EXPERIMENTOS = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN_EXPERIMENTOS}"
RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/{ESQUEMA}/{VOLUMEN}"
RUTA_TRAIN = f"{RUTA_VOLUMEN}/train.csv"
RUTA_TEST = f"{RUTA_VOLUMEN}/test.csv"

TABLA_BRONZE_TRAIN = f"{CATALOGO}.{ESQUEMA}.titanic_bronze_train"
TABLA_BRONZE_TEST = f"{CATALOGO}.{ESQUEMA}.titanic_bronze_test"

print("Catálogo :", CATALOGO)
print("Esquema  :", ESQUEMA)
print("Volume   :", RUTA_VOLUMEN)

## 3. Crea catálogo, esquema y Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN_EXPERIMENTOS}")


spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

print("Catálogo, esquema y Volume listos.")

## 4. Verificar archivos  en el Volume

In [0]:
archivos = [f.name for f in dbutils.fs.ls(RUTA_VOLUMEN)]
print("Archivos en el Volume:", archivos)

faltantes = [n for n in ["train.csv", "test.csv"] if n not in archivos]
assert not faltantes, (
    f"Faltan archivos en {RUTA_VOLUMEN}: {faltantes}. "
)
print("Ambos CSV están presentes.")

## 5. Ingesta a capa bronze

Definimos el esquema a mano para que los tipos sean deterministas y la ingesta sea reproducible. `train.csv` incluye `Survived`; `test.csv` no. Por eso construimos dos esquemas: uno completo y otro sin la etiqueta.

In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, DoubleType, StringType,
)

esquema_train = StructType([
    StructField("PassengerId", IntegerType(), True),
    StructField("Survived", IntegerType(), True),
    StructField("Pclass", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Sex", StringType(), True),
    StructField("Age", DoubleType(), True),
    StructField("SibSp", IntegerType(), True),
    StructField("Parch", IntegerType(), True),
    StructField("Ticket", StringType(), True),
    StructField("Fare", DoubleType(), True),
    StructField("Cabin", StringType(), True),
    StructField("Embarked", StringType(), True),
])

# El esquema de test es el mismo pero sin la columna Survived.
esquema_test = StructType(
    [c for c in esquema_train.fields if c.name != "Survived"]
)

df_train = (
    spark.read
    .option("header", True)
    .schema(esquema_train)
    .csv(RUTA_TRAIN)
)

df_test = (
    spark.read
    .option("header", True)
    .schema(esquema_test)
    .csv(RUTA_TEST)
)

## 6. Persistir como  Delta tables

Usamos `mode("overwrite")` para que la tabla se reemplace

In [0]:
(df_train.write.mode("overwrite").saveAsTable(TABLA_BRONZE_TRAIN))
(df_test.write.mode("overwrite").saveAsTable(TABLA_BRONZE_TEST))

n_train = spark.table(TABLA_BRONZE_TRAIN).count()
n_test = spark.table(TABLA_BRONZE_TEST).count()

print(f"{TABLA_BRONZE_TRAIN}: {n_train} filas")
print(f"{TABLA_BRONZE_TEST}:  {n_test} filas")

# Verificación de la predicción: 891 filas en train, 418 en test.
assert n_train == 891, f"Se esperaban 891 filas en train, se obtuvieron {n_train}"
assert n_test == 418, f"Se esperaban 418 filas en test, se obtuvieron {n_test}"
print("Conteos correctos: 891 en train, 418 en test.")

## 7. Conteo de nulos por columna (train)

Este diagnóstico guía la estrategia de imputación de la parte 82.

In [0]:
from pyspark.sql import functions as F

df_bronze_train = spark.table(TABLA_BRONZE_TRAIN)

exprs = [
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_bronze_train.columns
]
nulos = df_bronze_train.select(*exprs)
nulos.display()

## 8. Mostrar datos crudos

In [0]:
spark.table(TABLA_BRONZE_TRAIN).limit(10).display()